# Query the consolidated data
Load `all_consolidated.csv`, drop the redundant whole-sample aggregates so only
the granular data remains, then select and show data SQL-style (by scale,
subscale, item, sample_type, subsample, data_type).

The query logic lives in `src/query.py` (readable functions); this notebook
just drives it. Run the cells top to bottom.

## 1. Setup: imports and load the data

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))   # so we can import query.py

import pandas as pd
from query import select, distinct, overview

# Show all columns when printing.
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent
df_all = pd.read_csv(ROOT / "data" / "all_consolidated_corr.csv")
print(f"loaded {len(df_all)} rows, {len(df_all.columns)} columns")

#### Any column name of the data frame works as an input value for the select function:

In [ ]:
help(select)

## 2. Drop the redundant aggregates
`redundant_aggregate == True` marks whole-sample values that overlap their own
subsamples. We remove them so aggregation/selection uses only the granular
(non-overlapping) data. Everything below works on `df`.

In [ ]:
# Keep only the rows that are NOT redundant aggregates.
df = df_all[df_all["redundant_aggregate"] == False]
print(f"kept {len(df)} of {len(df_all)} rows after dropping redundant aggregates")

## 3. Orientation: what's in the table?
Before querying, see how many distinct values each key column has, and list the
values you might filter on.

In [ ]:
overview(df)

In [ ]:
# List the distinct values of any column you want to filter on:
print("scales:     ", distinct(df, "scale"))
print("subscales:  ", distinct(df, "subscale"))
print("record types:", distinct(df, "record_type"))
print("sample types:", distinct(df, "sample_type"))
print("data types: ", distinct(df, "data_type"))

## 4. Select data (SQL-style)
`select(df, column=value, ...)` returns the rows matching ALL the conditions
(an AND). A list value means "any of these" (like SQL's `IN`).
The result is a normal dataframe you can keep using.

**Example A — by scale + data_type**  
`SELECT * WHERE scale='DES_T' AND data_type='mean'`

In [ ]:
select(df, scale="DES_T", data_type="mean")

**Example B — by sample_type + subsample**  
`SELECT * WHERE sample_type='patients' AND subsample='taxon'`

In [ ]:
select(df, sample_type="patients", subsample="taxon")

**Example C — by subscale** (e.g. a CTQ subscale)  
`SELECT * WHERE subscale='EA'`

In [ ]:
df_ea = select(df, subscale="EA")
df_ea.head()

In [ ]:
# Check scale (only CTQ_SF should be among this data selection: 
set(df_ea.scale)

**Example D — items only**  
`SELECT * WHERE record_type='item'`

In [ ]:
df_items = select(df, record_type="item")
df_items.head()

In [ ]:
# Check publications (only three publications with individual items have been loaded so far:
set(df_items.publication)


**Example E — several data_types at once** (IN-style)  
`SELECT * WHERE scale='DES_T' AND data_type IN ('mean','median')`

In [ ]:
df_dest_t_mn_md = select(df, scale="DES_T", data_type=["mean", "median"])
df_dest_t_mn_md.head()

In [ ]:
# Check publications (so far only 4 DES-T publications have been used:
set(df_dest_t_mn_md.publication)


In [ ]:
set(df_dest_t_mn_md.scale)

**Example F — combine many conditions**  
`SELECT * WHERE scale='DES_T' AND sample_type='patients' AND data_type='mean'`

In [ ]:
select(df, scale="DES_T", sample_type="patients", data_type="mean")

## 5. Keep working with a result
Because `select` returns a dataframe, you can store it and use it further —
e.g. just the `value` column, or a spot-check subset.

In [ ]:
# Store a selection, then use it like any dataframe.
des_t_patient_means = select(df, scale="DES_T", sample_type="patients", data_type="mean")

# e.g. just the values:
print("values:", des_t_patient_means["value"].tolist())

# e.g. narrow further by hand:
des_t_patient_means[des_t_patient_means["subsample"] == "taxon"]

In [ ]:
df[(df.publication == 'Modestin & Erni 2004') & (df.data_type == 'mean')]

## 6. Check with a concrete example (DES-T):
- Verify if functions work as expected, aggregate over different subsamples for DES-T.
- Get mean of means values as gauge of what values to expect.


In [ ]:
def aggregate_over_subsamples(df, data_type="mean"):
    """Weighted mean across subsamples, giving one value per
    scale / subscale / record_type / sample_type.

    Pools a sample_type's subsamples together (drops `subsample` from the
    grouping), weighting each by its sample_size:
        weighted = sum(value * n) / sum(n)

    Assumes redundant aggregates have already been removed, so the subsamples
    don't overlap. Only `mean`/`median` should be passed (averaging SDs is
    not valid).
    """
    # keep only the statistic we want (e.g. the means)
    subset = df[df["data_type"] == data_type].copy()

    # make sure value and weight are numeric
    subset["value"] = pd.to_numeric(subset["value"], errors="coerce")
    subset["sample_size"] = pd.to_numeric(subset["sample_size"], errors="coerce")
    subset = subset.dropna(subset=["value", "sample_size"])

    # group WITHOUT subsample -> pools subsamples together
    group_cols = ["scale", "subscale", "record_type", "sample_type"]

    results = []
    for key, g in subset.groupby(group_cols):
        weighted = (g["value"] * g["sample_size"]).sum() / g["sample_size"].sum()
        row = dict(zip(group_cols, key))
        row["weighted_value"] = weighted
        row["n_groups"] = len(g)
        row["total_n"] = g["sample_size"].sum()
        results.append(row)

    return pd.DataFrame(results)

In [ ]:
from aggregate import aggregate

### Aggregate the DES-T means for all subsamples from all publications:

In [ ]:
# Select DES-T means: 
des_t_means = select(df, scale="DES_T", data_type="mean")
set(des_t_means.sample_type)

In [ ]:
# Check if dimensions are as expected:
print(des_t_means.shape)
print(df[(df.scale=="DES_T") & (df.data_type=="mean")].shape)

In [ ]:
# Check subsamples:
set(des_t_means.subsample)